# Catalogue metadata that fits the card, in every Indian language

**This notebook has not been run.** Every code cell ships with an empty output because there was
no API key available when it was written. Nothing here is a recorded result; run it yourself with
your own key to see real output.

A catalogue row has four text fields and each one has to fit a box on a card. The check everybody
writes is `len(text) <= 40`. In English that is right. In Devanagari, Telugu, Tamil, Bengali,
Kannada, Malayalam, Gujarati, Gurmukhi or Odia it is wrong in two ways at once, and this notebook
walks through both of them.

Pipeline overview:

1. Measure the English source row against its budgets, counting visible characters, not codepoints
2. Translate the row with `sarvam-translate:v1`
3. Measure the translation the same way, and see where `len()` would have disagreed
4. Ask `sarvam-105b` for a shorter phrasing of anything that overflows
5. Cut anything that still overflows, on a character boundary, and save the report

The first step and the last need no key. Steps 2 to 4 do.

In [ ]:
%pip install -r requirements.txt

## Setup

The key is passed to the client **explicitly**. The SDK's constructor takes it as a default
argument, which Python evaluates once, when the SDK is first imported — so calling `load_dotenv()`
after that import is too late and the constructor raises.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

# The three modules sit beside this notebook.
sys.path.insert(0, str(Path.cwd()))

from fit_gate import DEMO_BUNDLE, FIELD_BUDGETS, OVER, lint_bundle, render_report
from grapheme_clusters import cluster_count, cluster_safe_truncate
from sarvam_metadata import rewrite_to_fit, translate_bundle

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or in a .env file. "
        "Copy .env.example to .env and put your key in it."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. The two numbers

`len()` counts codepoints. A reader counts visible characters. These are the same number in
English and they are not the same number anywhere else.

In [ ]:
word = "क्षेत्र"          # "kshetra", region

print("codepoints:", len(word))
print("what a reader counts:", cluster_count(word))
print("the pieces:", [hex(ord(c)) for c in word])

The same word, spelled the two ways Unicode allows, makes the point harder. `क़लम` ("qalam", pen)
can be written with a single precomposed letter or with a base letter plus a nukta mark. The two
spellings are different strings with different `len()` values, and a reader sees the same three
characters either way.

In [ ]:
precomposed = "\u0958\u0932\u092e"              # qalam, one codepoint for the first letter
decomposed = "\u0915\u093c\u0932\u092e"        # qalam, base letter plus a nukta

print("same string?      ", precomposed == decomposed)
print("same len()?       ", len(precomposed), len(decomposed))
print("same count?       ", cluster_count(precomposed), cluster_count(decomposed))

## 2. The naive slice

Once the `len()` check says "over by 9", somebody writes `text[:90]`. That cuts wherever it lands,
and on Indian text about a third of the landing places are inside a character. Below, the naive
slice severs a vowel sign off its consonant and leaves it hanging at the start of the remainder.

In [ ]:
hindi = "पुणे की एक हाउसिंग कॉलोनी में दो ऊबे हुए चचेरे भाई एक गुम हुई साइकिल को अपना पहला केस बना लेते हैं।"

naive = hindi[:12]
safe = cluster_safe_truncate(hindi, 12)

print("naive slice :", repr(naive))
print("what is left:", repr(hindi[12:17]), "<- starts with an orphaned vowel sign")
print("cluster safe:", repr(safe))
print("visible characters kept:", cluster_count(safe))

`cluster_safe_truncate` pays for the ellipsis out of the budget rather than adding it on top, so
the result is never longer than asked for. When the budget is too small to hold both, the ellipsis
is dropped instead of the budget being broken.

In [ ]:
for budget in (1, 2, 3, 12, 22, 40):
    cut = cluster_safe_truncate(hindi, budget)
    print(f"budget {budget:>3}  ->  {cluster_count(cut):>3} characters  {cut}")

## 3. The gate on the English source

`lint_bundle` measures every field against its budget and `render_report` prints both numbers side
by side. The budgets in `fit_gate.py` are **demo values**, not any platform's real limits.

In [ ]:
print("budgets, in visible characters:", dict(FIELD_BUDGETS))
print()
print(render_report(lint_bundle(DEMO_BUNDLE)))

Two of the four English fields overflow. That is worth sitting with for a second: the copy that
was written to fit is the copy that does not.

## 4. Translate the row

`sarvam-translate:v1` in formal mode. It covers all 22 scheduled languages, and formal is the only
mode it supports.

Note the argument name: translation takes `target_language_code`. Text to speech takes
`language_code`. Both spellings exist in the SDK and swapping them is a real bug people ship.

In [ ]:
TARGET = "hi-IN"

translated = translate_bundle(client, DEMO_BUNDLE, TARGET)

for field, text in translated.items():
    print(f"{field}: {text}")

## 5. Measure the translation, both ways

This is the cell that shows why the recipe exists. For each field it prints what `len()` would
have decided and what counting visible characters decides.

In [ ]:
print(render_report(lint_bundle(translated)))
print()
print("where the two checks disagree:")
for field, text in translated.items():
    budget = FIELD_BUDGETS[field]
    naive_fits = len(text) <= budget
    real_fits = cluster_count(text) <= budget
    if naive_fits != real_fits:
        print(f"  {field}: len()={len(text)} says over, "
              f"visible characters={cluster_count(text)} says it fits")

## 6. Shorten anything that still overflows

`rewrite_to_fit` asks the chat model for a shorter phrasing in the same language, measures the
reply, and returns the first one that fits. The loop is bounded at three attempts. If none of them
fits, the shortest candidate is cut to the budget and the result is marked `fell_back`, so you can
tell a real rewrite from a machine cut.

In [ ]:
fixed = dict(translated)

for verdict in lint_bundle(translated):
    if verdict.verdict != OVER:
        continue
    result = rewrite_to_fit(
        client, verdict.text, verdict.field, verdict.budget, TARGET
    )
    fixed[verdict.field] = result.text
    print(f"{verdict.field}: {verdict.clusters} -> {cluster_count(result.text)} characters "
          f"in {result.attempts} attempt(s), fell back: {result.fell_back}")
    print(f"  {result.text}")

## 7. The final report

With `truncate=True` the gate also shows what a cluster-safe cut to the budget would look like.
The previews are printed under the table, never inside it: a plain text table cannot line up
Indian scripts, because a count of visible characters is not a display width.

In [ ]:
report = render_report(lint_bundle(fixed, truncate=True))
print(report)

report_path = OUTPUT_DIR / f"fit_report_{TARGET}.txt"
report_path.write_text(report, encoding="utf-8")
print()
print("written to", report_path)

## What this does not do

- **Pixels.** Counting visible characters is a much better proxy than counting codepoints and it
  is still a proxy. A Devanagari character is wider on screen than a Latin one, and a real card
  limit is a width in pixels against a particular font.
- **Full UAX #29.** The segmenter is an approximation of it, using the standard library only. The
  five cases where it knowingly differs are listed in `UNSUPPORTED_FEATURES` in
  `grapheme_clusters.py` and in the README. All five over-count, so they make the gate stricter
  than reality rather than laxer.
- **Right-to-left text.** Urdu is in the translation model's language list and laying it out
  raises a different set of questions. Not attempted here.

`grapheme_clusters.py` and `fit_gate.py` import nothing but the standard library, so you can lift
either of them into your own project and test them without a network.